# SIPM: Strongly Convex Example

A minimal strongly convex example with `mu`-strong convexity and a single linear constraint `x[0] <= 0`.

In [2]:
import math
import torch
from sipm import make_sampler, SIPMConfig, StochasticInexactPenaltyMethod, constant_schedule

def sampler(batch_size: int):
    zeta = torch.zeros(batch_size, 1)
    xi = torch.zeros(batch_size, 1)
    return zeta, xi

def constraint_map(xi_batch):
    batch = xi_batch.shape[0]
    n = 3
    a = torch.zeros(batch, 1, n)
    a[:, 0, 0] = 1.0
    b = torch.zeros(batch, 1)
    return a, b

mu = 2.0
c = torch.tensor([1.5, -2.0, 0.5])
def objective(x, zeta_batch):
    batch = zeta_batch.shape[0]
    loss = 0.5 * mu * (x - c).pow(2).sum()
    return loss.repeat(batch)

def step_size(k: int) -> float:
    if k == 0:
        return 2.0 / mu
    return 2.0 / (mu * k)

def penalty_weight(k: int) -> float:
    return math.log(k + 1.0)

def delta(k: int) -> float:
    if k == 0:
        return 1.0
    return 1.0 / k

cfg = SIPMConfig(
    num_iters=500,
    step_size=step_size,
    penalty_weight=penalty_weight,
    delta=delta,
    iterate_weight=lambda k: 1.0 / step_size(k),
    batch_size=1,
)

solver = StochasticInexactPenaltyMethod(
    objective=objective,
    constraint_map=constraint_map,
    sampler=make_sampler(sampler),
    config=cfg,
)

x0 = torch.zeros(3)
x_bar, history = solver.run(x0)
print("x_bar:", x_bar)
print("final penalty:", history["penalty"][-1])

x_bar: tensor([ 1.9257e-03, -2.0000e+00,  5.0000e-01])
final penalty: 0.0019061543280258775
